# FIT4112 · Lab 1 · Notebook 03 — Apriori và luật kết hợp

**Mục tiêu:** tìm sản phẩm thường được mua cùng nhau và giải thích ba chỉ số:

- **Support(A → B):** tỷ lệ hóa đơn có cả A và B.
- **Confidence(A → B):** trong các hóa đơn có A, tỷ lệ có cả B.
- **Lift(A → B):** mức độ đồng xuất hiện của A và B so với trường hợp độc lập; **lift > 1** gợi ý mối liên hệ dương, không chứng minh nhân quả.

**Đầu vào:** `basket_bool.csv` từ notebook 02. **Đầu ra:** `association_rules.csv`.

In [ ]:
from pathlib import Path
import pandas as pd

def find_project_root():
    """Tìm thư mục gốc dù notebook chạy từ root hay notebooks/."""
    cwd = Path.cwd().resolve()
    for folder in [cwd, *cwd.parents]:
        if (folder / "data" / "raw" / "transactions_demo.csv").exists():
            return folder
    raise FileNotFoundError("Không tìm thấy data/raw/transactions_demo.csv. Hãy clone đúng repo.")

ROOT = find_project_root()
RAW_DIR = ROOT / "data" / "raw"
OUT_DIR = ROOT / "data" / "processed"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Project root:", ROOT)
from mlxtend.frequent_patterns import apriori, association_rules
import matplotlib.pyplot as plt

## 1. Tham số dễ thử

**Nhiệm vụ B:** chạy hai lần với `MIN_SUPPORT = 0.10` và `0.20`, ghi lại số tập mục và luật. Nếu dùng dữ liệu thật với quá ít luật, giảm min_support có kiểm soát.

In [ ]:
MIN_SUPPORT = 0.10       # TODO: đổi thành 0.20 và so sánh
MIN_CONFIDENCE = 0.30
MAX_LEN = 3
basket_file = OUT_DIR / "basket_bool.csv"
if not basket_file.exists():
    raise FileNotFoundError("Hãy chạy notebook 02 để tạo basket_bool.csv")
basket = pd.read_csv(basket_file, index_col=0).astype(bool)
print("Kích thước basket:", basket.shape)
display(basket.head().astype(int))

## 2. Apriori – khai phá tập mục phổ biến

In [ ]:
frequent = apriori(basket, min_support=MIN_SUPPORT, use_colnames=True, max_len=MAX_LEN)
print("Số tập mục phổ biến:", len(frequent))
display(frequent.sort_values("support", ascending=False).head(10))
if frequent.empty:
    raise ValueError("Không có tập mục phổ biến; hãy giảm MIN_SUPPORT.")

## 3. Sinh luật kết hợp và lọc theo confidence

In [ ]:
if not (frequent["itemsets"].map(len) >= 2).any():
    raise ValueError("Không có tập mục từ 2 sản phẩm; hãy giảm MIN_SUPPORT.")
rules = association_rules(frequent, metric="confidence", min_threshold=MIN_CONFIDENCE)
rules = rules.copy()
rules["antecedents_str"] = rules["antecedents"].map(lambda s: ", ".join(sorted(s)))
rules["consequents_str"] = rules["consequents"].map(lambda s: ", ".join(sorted(s)))
selected_cols = ["antecedents_str", "consequents_str", "support", "confidence", "lift"]
rules_view = rules[selected_cols].sort_values(["lift", "confidence"], ascending=False)
print("Số luật:", len(rules_view))
display(rules_view.head(15).round(3))

## 4. Quan sát và lưu kết quả

**Quan trọng:** luật có lift cao nhưng support rất thấp có thể ít hữu ích trong kinh doanh. Nên xem cả ba chỉ số.

In [ ]:
if not rules_view.empty:
    top = rules_view.head(10).copy()
    labels = top["antecedents_str"] + " → " + top["consequents_str"]
    ax = top.set_index(labels)["lift"].sort_values().plot.barh(figsize=(9, 5), title="Top luật theo Lift")
    ax.set_xlabel("Lift")
    plt.tight_layout()
    plt.show()
else:
    print("Không có luật đạt MIN_CONFIDENCE. Thử ngưỡng thấp hơn.")

out_file = OUT_DIR / "association_rules.csv"
rules_view.to_csv(out_file, index=False)
print("Đã lưu:", out_file)

## 5. Bài tập (viết câu trả lời vào báo cáo)

**TODO 1:** Chọn 5 luật từ `rules_view`, ghi support/confidence/lift và sản phẩm tương ứng.

**TODO 2:** Chạy `MIN_SUPPORT = 0.10` rồi `0.20`. So sánh số tập mục phổ biến và số luật.

**TODO 3:** Giải thích 2 luật bằng lời của mình. **Không diễn giải tương quan thành quan hệ nhân quả.**

**TODO 4:** Đề xuất 1 chiến lược bán kèm/trưng bày và nêu 1 hạn chế của bộ dữ liệu.

Tham khảo [phiếu thực hành](../docs/LAB01_STUDENT.md) và [mẫu báo cáo](../docs/REPORT_TEMPLATE.md).